# Sólidos Primitivos e Operações Booleanas
## CadQuery para Arquitetos e Engenheiros
### Versão VS Code

---

Neste notebook vamos explorar as **formas primitivas** do CadQuery e as **operações de modelagem** que permitem criar geometrias arquitetônicas mais complexas.

---

## Importações

In [ ]:
import cadquery as cq
from ocp_vscode import show, set_port, show_object

In [ ]:
set_port(3939)

---

## Sólidos Primitivos

Os **sólidos primitivos** são formas geométricas básicas que o CadQuery consegue criar diretamente. São o ponto de partida para construções mais complexas.

---

### Caixa — `.box()`

A caixa retangular é a primitiva mais usada em arquitetura. Representa qualquer volume ortogonal: paredes, lajes, pilares retangulares, blocos.

```python
.box(length, width, height)
```

| Parâmetro | Eixo | Descrição |
|-----------|------|-----------|
| `length` | X | Comprimento |
| `width`  | Y | Largura |
| `height` | Z | Altura |

Por padrão, a caixa é criada **centrada na origem** do Workplane. O parâmetro `centered` controla esse comportamento:
- `centered=True` → centro geométrico na origem (padrão)
- `centered=False` → canto inferior esquerdo na origem

In [ ]:
# Caixa centrada na origem (padrão)
pilar_centrado = cq.Workplane("XY").box(0.4, 0.4, 3.0)

# Caixa com canto inferior esquerdo na origem
pilar_canto = cq.Workplane("XY").box(0.4, 0.4, 3.0, centered=False)

show(
    pilar_centrado,
    pilar_canto,
    names=["Centrado", "Canto na origem"]
)

---

### Cilindro — `.cylinder()`

O cilindro representa pilares circulares, colunas, tanques, torres cilíndricas.

```python
.cylinder(height, radius, angle=360)
```

| Parâmetro | Descrição |
|-----------|-----------|
| `height` | Altura |
| `radius` | Raio da base |
| `angle`  | Ângulo de abertura em graus — padrão `360` (cilindro completo) |

O parâmetro `angle` permite criar **segmentos de cilindro**, úteis para curvas parciais e elementos curvos.

In [ ]:
# Coluna circular completa
coluna = cq.Workplane("XY").cylinder(3.0, 0.2)

# Parede curva: segmento de cilindro de 90°
parede_curva = cq.Workplane("XY").cylinder(2.8, 1.5, angle=90)

# Parede curva em semicírculo: 180°
parede_semi = cq.Workplane("XY").cylinder(2.8, 2.5, angle=180)

show(
    coluna,
    parede_curva,
    parede_semi,
    names=["Coluna", "Parede curva 90°", "Parede semicircular"]
)

---

### Esfera — `.sphere()`

A esfera é usada em cúpulas, elementos decorativos, volumes orgânicos e estudo de formas.

```python
.sphere(radius, angle1=-90, angle2=90, angle3=360)
```

| Parâmetro | Descrição |
|-----------|-----------|
| `radius`  | Raio |
| `angle1`  | Ângulo inicial na vertical (latitude inferior) — padrão `-90°` |
| `angle2`  | Ângulo final na vertical (latitude superior) — padrão `90°` |
| `angle3`  | Ângulo de abertura horizontal — padrão `360°` |

Controlando `angle1` e `angle2` é possível criar **calotas esféricas** — como cúpulas.

In [ ]:
# Esfera completa
esfera = cq.Workplane("XY").sphere(2.0)

# Calota superior: metade de cima da esfera
cupula = cq.Workplane("XY").sphere(2.0, angle1=0, angle2=90)

# Calota rasa: apenas o terço superior
cupula_rasa = cq.Workplane("XY").sphere(2.0, angle1=60, angle2=90)

show(
    esfera,
    cupula,
    cupula_rasa,
    names=["Esfera completa", "Cúpula", "Cúpula rasa"]
)

---

### Cone e Tronco de Cone — `BRepPrimAPI_MakeCone`

O cone e o tronco de cone são criados diretamente via OCCT, da mesma forma que o toro:

```python
from OCP.BRepPrimAPI import BRepPrimAPI_MakeCone

# BRepPrimAPI_MakeCone(raio_base, raio_topo, altura)
# raio_topo = 0  → cone pontiagudo
# raio_topo > 0  → tronco de cone
solido = cq.Workplane().add(cq.Shape(BRepPrimAPI_MakeCone(r1, r2, h).Shape()))
```

In [ ]:
from OCP.BRepPrimAPI import BRepPrimAPI_MakeCone

# --- Cone pontiagudo ---
# raio_base=2.5, raio_topo=0, altura=2.0
telhado_conico = cq.Workplane().add(cq.Shape(BRepPrimAPI_MakeCone(2.5, 0, 2.0).Shape()))

# --- Tronco de cone — base maior em baixo ---
# raio_base=3.0, raio_topo=1.5, altura=3.0
tronco = cq.Workplane().add(cq.Shape(BRepPrimAPI_MakeCone(3.0, 1.5, 3.0).Shape()))
tronco = tronco.translate((8.0, 0, 0))

# --- Tronco invertido — base menor em baixo ---
# raio_base=1.0, raio_topo=3.0, altura=2.0
tronco_invertido = cq.Workplane().add(cq.Shape(BRepPrimAPI_MakeCone(1.0, 3.0, 2.0).Shape()))
tronco_invertido = tronco_invertido.translate((16.0, 0, 0))

show(
    telhado_conico,
    tronco,
    tronco_invertido,
    names=["Telhado cônico", "Tronco de cone", "Tronco invertido"]
)

---

### Superfície de Revolução — `.revolve()` e Toro

O **revolve** (revolução) cria um sólido girando um perfil 2D em torno de um eixo. É a operação adequada para qualquer forma com **simetria axial**: coberturas anulares, colunas com entases, formas de vaso.

O processo é sempre o mesmo:
1. Escolher um plano de trabalho que contenha o perfil e o eixo de revolução
2. Desenhar o perfil deslocado do eixo
3. Chamar `.revolve(360)` — o eixo padrão do `Workplane("XZ")` é o eixo Z do mundo

**Limitação do OCCT**: revolucionar um perfil **circular** 360° em torno de um eixo coplanar (o caso do toro) causa uma falha numérica interna do motor geométrico. Para criar um toro, usamos diretamente a primitiva `BRepPrimAPI_MakeTorus` do OCCT, que é o que o CadQuery usava internamente em versões anteriores:

```python
from OCP.BRepPrimAPI import BRepPrimAPI_MakeTorus

# r1 = raio do anel central, r2 = raio do tubo
anel = cq.Workplane().add(cq.Shape(BRepPrimAPI_MakeTorus(r1, r2).Shape()))
```

In [ ]:
from OCP.BRepPrimAPI import BRepPrimAPI_MakeTorus

# --- Toro (argola) via BRepPrimAPI_MakeTorus ---
# r1 = raio do anel central (distância do centro ao tubo)
# r2 = raio do tubo (espessura do anel)
r1 = 3.0
r2 = 0.5

anel = cq.Workplane().add(cq.Shape(BRepPrimAPI_MakeTorus(r1, r2).Shape()))

# --- Anel com tubo mais espesso ---
r1_grosso = 3.0
r2_grosso = 1.2

anel_grosso = cq.Workplane().add(cq.Shape(BRepPrimAPI_MakeTorus(r1_grosso, r2_grosso).Shape()))

show(
    anel,
    anel_grosso,
    names=["Anel fino", "Anel grosso"]
)

O revolve não precisa ser circular. Qualquer perfil 2D fechado pode ser girado em torno de um eixo. O exemplo abaixo cria um **vaso** usando um perfil retangular arredondado:

In [ ]:
# Perfil trapezoidal girado 360° → tronco de cone via revolve
# Demonstra que qualquer perfil pode ser revolvido

dist_eixo = 0.5   # distância mínima do perfil ao eixo
larg_base = 2.0   # largura da base do perfil
larg_topo = 1.0   # largura do topo do perfil
altura_p  = 3.0   # altura do perfil

forma_revolvida = (
    cq.Workplane("XZ")
    .polyline([
        (dist_eixo,             0),
        (dist_eixo + larg_base, 0),
        (dist_eixo + larg_topo, altura_p),
        (dist_eixo,             altura_p),
    ])
    .close()
    .revolve(360)
)

show(forma_revolvida, names=["Forma revolvida"])

---

### Rampa — via Loft entre Perfis Paralelos

O loft entre **dois perfis idênticos em posições diferentes** cria uma transição inclinada — ideal para rampas. O perfil do topo é uma cópia da base deslocada para a outra extremidade e elevada:

In [ ]:
# --- Rampa via loft ---
from copy import copy

dx_r = 5.0   # comprimento da rampa
dy_r = 3.0   # largura da rampa
dz_r = 1.5   # altura total da rampa

# Perfil da base: retângulo em uma extremidade
base_rampa = cq.Wire.makePolygon([
    cq.Vector(-dx_r/2, -dy_r/2, 0),
    cq.Vector(-dx_r/2,  dy_r/2, 0),
    cq.Vector(-dx_r/2,  dy_r/2, 0),
    cq.Vector(-dx_r/2, -dy_r/2, 0),
    cq.Vector(-dx_r/2, -dy_r/2, 0),
])

# Perfil do topo: cópia da base deslocada para a outra extremidade e elevada
topo_rampa = copy(base_rampa).translate((dx_r, 0, dz_r))

rampa = (
    cq.Workplane("XY")
    .add(base_rampa)
    .add(topo_rampa)
    .toPending()
    .loft()
)

show(rampa, names=["Rampa"])

---

## Operações Booleanas

As **operações booleanas** permitem combinar dois ou mais sólidos para criar formas que não seriam possíveis com uma única primitiva.

| Operação | Método | Resultado |
|----------|--------|-----------|
| União | `.union(outro)` | Une os dois sólidos em um único volume |
| Subtração | `.cut(outro)` | Remove o volume do segundo sólido do primeiro |
| Interseção | `.intersect(outro)` | Mantém apenas o volume compartilhado pelos dois |

> 💡 **Dica**: nas operações booleanas, a **ordem importa** para subtração e interseção. `a.cut(b)` é diferente de `b.cut(a)`.

---

### União — `.union()`

A união **funde** dois sólidos em um único objeto. As superfícies internas de contato são eliminadas, resultando em um volume contínuo.

Contexto arquitetônico: unir o corpo de um edifício com uma marquise, uma escada com um patamar, ou dois volumes de uma composição.

In [ ]:
# Corpo principal do edifício
corpo = cq.Workplane("XY").box(10.0, 8.0, 6.0)

# Volume de acesso — conectado ao corpo principal
acesso = cq.Workplane("XY", origin=(7.0, 0, -1.5)).box(4.0, 4.0, 3.0)

# Antes da união: os dois volumes separados
show(
    corpo,
    acesso,
    names=["Corpo", "Acesso"]
)

In [ ]:
# Após a união: um único sólido contínuo
edificio = corpo.union(acesso)

show(edificio, names=["Edifício unido"])

---

### Subtração — `.cut()`

A subtração **remove** do primeiro sólido o volume ocupado pelo segundo. O sólido subtraído funciona como um "molde" negativo.

Contexto arquitetônico: abrir vãos em paredes, criar janelas e portas, escavar terraços, abrir pátios em edifícios.

In [ ]:
# Parede sólida
parede = cq.Workplane("XY").box(8.0, 0.3, 3.0)

# Vão da janela — posicionado no centro da parede
vao_janela = cq.Workplane("XY", origin=(0, 0, 0.4)).box(1.5, 0.5, 1.5)

# Vão da porta — posicionado à esquerda
vao_porta = cq.Workplane("XY", origin=(-3.0, 0, -0.5)).box(1.0, 0.5, 2.0)

# Abrindo os vãos na parede com duas subtrações encadeadas
parede_com_vaos = parede.cut(vao_janela).cut(vao_porta)

show(parede_com_vaos, names=["Parede com vãos"])

In [ ]:
# Exemplo 2: pátio escavado em um volume compacto

# Volume total do edifício
bloco = cq.Workplane("XY").box(12.0, 12.0, 4.0)

# Pátio interno — um volume subtraído do centro
patio = cq.Workplane("XY", origin=(0, 0, 0.5)).box(6.0, 6.0, 5.0)

edificio_patio = bloco.cut(patio)

show(edificio_patio, names=["Edifício com pátio"])

---

### Interseção — `.intersect()`

A interseção **mantém apenas o volume que os dois sólidos compartilham**.

Contexto arquitetônico: encontrar o volume máximo dentro de duas restrições simultâneas, como um envelope de gabarito e um cone de afastamento.

In [ ]:
# Exemplo: cruzamento de dois volumes cilíndricos

cilindro_h = cq.Workplane("XZ").cylinder(6.0, 2.0)
cilindro_v = cq.Workplane("XY").cylinder(6.0, 2.0)

# Antes da interseção
show(
    cilindro_h,
    cilindro_v,
    names=["Cilindro horizontal", "Cilindro vertical"]
)

In [ ]:
# Resultado da interseção: apenas o volume compartilhado
intersecao = cilindro_h.intersect(cilindro_v)

show(intersecao, names=["Interseção"])

In [ ]:
# Exemplo arquitetônico: envelope máximo x cone de afastamento (via loft)

# Envelope cúbico: volume máximo permitido pela legislação
envelope = cq.Workplane("XY").box(10.0, 10.0, 8.0)

# Cone de afastamento via loft: transição de círculo grande (base) para pequeno (topo)
base_af = cq.Wire.makeCircle(14.0, cq.Vector(0, -7, -4), cq.Vector(0, 0, 1))
topo_af = cq.Wire.makeCircle(2.0,  cq.Vector(0, -7, 8),  cq.Vector(0, 0, 1))

cone_afastamento = cq.Workplane("XY").add(base_af).add(topo_af).toPending().loft()

show(
    envelope,
    cone_afastamento,
    names=["Envelope", "Cone de afastamento"]
)

In [ ]:
volume_resultante = envelope.intersect(cone_afastamento)

show(volume_resultante, names=["Volume resultante"])

---

## Combinando Operações

Na prática, projetos arquitetônicos combinam várias operações em sequência. Veja um exemplo: um edifício com torre, cobertura cônica e aberturas.

In [ ]:
# --- Dimensões ---
base_larg  = 10.0
base_prof  = 8.0
base_alt   = 4.0
torre_raio = 1.5
torre_alt  = 6.0

# --- Volumes base e torre ---
base  = cq.Workplane("XY").box(base_larg, base_prof, base_alt)
torre = cq.Workplane("XY").cylinder(torre_alt, torre_raio)

# --- Cobertura cônica via BRepPrimAPI_MakeCone ---
z_cobertura = torre_alt / 2
cobertura = cq.Workplane().add(cq.Shape(BRepPrimAPI_MakeCone(2.0, 0.01, 2.5).Shape()))
cobertura = cobertura.translate((0, 0, z_cobertura))

# --- Vãos das janelas ---
janela_frente = cq.Workplane("XY", origin=(0, 0, 0.5)).box(2.0, base_prof + 0.5, 1.5)
janela_lado   = cq.Workplane("XY", origin=(0, 0, 0.5)).box(base_larg + 0.5, 2.0, 1.5)

# --- Montagem ---
edificio = base.union(torre)
edificio = edificio.union(cobertura)
edificio = edificio.cut(janela_frente)
edificio = edificio.cut(janela_lado)

show(edificio, names=["Composição final"])

---

## Exercício

Crie um modelo volumétrico de um **pavilhão simples** com os seguintes elementos:

1. **Base**: volume retangular representando as paredes
2. **Cobertura**: use loft para criar uma rampa de acesso
3. **Vãos**: abra pelo menos duas janelas e uma porta usando subtração
4. **Detalhe**: adicione colunas cilíndricas em pelo menos dois cantos usando união

Use variáveis para todas as dimensões e exiba o resultado final com `show()`.

In [ ]:
# Escreva seu código aqui



---

## Resumo

Neste notebook você aprendeu:

- As primitivas do CadQuery: `box`, `cylinder`, `sphere`
- O parâmetro `centered` da caixa e o parâmetro `angle` do cilindro e da esfera
- Como criar **cúpulas** com `sphere(angle1, angle2)`
- Como criar **cones e troncos** com `BRepPrimAPI_MakeCone` do OCCT
- Como criar **superfícies de revolução** com `.revolve()` — toros, vasos e formas axialmente simétricas
- Como criar **rampas** via **loft** entre perfis paralelos deslocados
- As três **operações booleanas**: `.union()`, `.cut()`, `.intersect()`
- Como **combinar** primitivas, loft, revolve e booleanas para construir modelos complexos

No próximo notebook veremos como trabalhar com **perfis 2D e extrusões** — a base da modelagem arquitetônica a partir de plantas baixas.

---
*CadQuery para Arquitetos — Notebook 2*